# Text-to-Speech Comparison: AudioLDM2 vs Bark vs TangoFlux

The focus of the assignment:
1. Compare pretrained text-to-speech and text-to-audio models on the same expressive prompt set.
2. Generate one audio sample per model for each prompt and store outputs in a consistent folder structure.
3. Evaluate generated audio using objective signal metrics, CLAP semantic alignment, and UTMOS speech naturalness.
4. Analyze trade-offs between prompt/style alignment and natural-sounding speech across AudioLDM2, Bark, and TangoFlux.

References:

- https://audioldm.github.io/
- https://huggingface.co/docs/diffusers/api/pipelines/audioldm2
- https://github.com/suno-ai/bark
- https://huggingface.co/spaces/declare-lab/TangoFlux

## 1. Dependency Setup

Install the project dependencies once before running the notebook:

```bash
pip install -r requirements.txt
```


In [6]:
import os, random
import warnings
from pathlib import Path
import gc

warnings.filterwarnings("ignore")

OUTPUT_DIR = Path("outputs")
NUMBA_CACHE_DIR = OUTPUT_DIR / "numba_cache"

OUTPUT_DIR.mkdir(exist_ok=True)
NUMBA_CACHE_DIR.mkdir(exist_ok=True)

os.environ.update({
    "HF_HUB_DISABLE_SYMLINKS_WARNING": "1",
    "HF_HUB_DISABLE_PROGRESS_BARS": "0",
    "TRANSFORMERS_VERBOSITY": "warning",
    "MPLBACKEND": "Agg",
    "NUMBA_CACHE_DIR": str(NUMBA_CACHE_DIR.resolve())
})

import espeakng_loader

os.environ["PHONEMIZER_ESPEAK_LIBRARY"] = espeakng_loader.get_library_path()
os.environ["ESPEAK_DATA_PATH"] = espeakng_loader.get_data_path()

import torch

import numpy as np
import pandas as pd
import matplotlib
import time

matplotlib.use("Agg", force=True)

import matplotlib.pyplot as plt
from IPython.display import Audio, Image, display

from phonemizer.backend import EspeakBackend
from transformers import AutoProcessor, BarkModel, GPT2Model, ClapModel, ClapProcessor

from tts_utils import MetricsCollector, normalize_audio, save_audio


def patch_gpt2_generation():
    def update_model_kwargs(self, outputs, model_kwargs):
        if getattr(outputs, "past_key_values", None) is not None:
            model_kwargs["past_key_values"] = outputs.past_key_values

        if "token_type_ids" in model_kwargs:
            token_type_ids = model_kwargs["token_type_ids"]
            model_kwargs["token_type_ids"] = torch.cat(
                [token_type_ids, token_type_ids[:, -1:]],
                dim=-1
            )

        if "attention_mask" in model_kwargs:
            attention_mask = model_kwargs["attention_mask"]
            model_kwargs["attention_mask"] = torch.cat(
                [attention_mask, attention_mask.new_ones((attention_mask.shape[0], 1))],
                dim=-1
            )

        return model_kwargs

    if not hasattr(GPT2Model, "_update_model_kwargs_for_generation"):
        GPT2Model._update_model_kwargs_for_generation = update_model_kwargs


patch_gpt2_generation()

if not EspeakBackend.is_available():
    raise RuntimeError("AudioLDM2 needs eSpeak NG. Install espeakng-loader or rerun the setup cell.")

print("Imports ready")

Imports ready


## 2. GPU/Reproducibility

In [7]:
REQUIRE_GPU = True
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.empty_cache()
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    message = "CUDA GPU is not available. Install a CUDA-enabled PyTorch build and check the NVIDIA driver."
    if REQUIRE_GPU:
        raise RuntimeError(message)
    DEVICE = "cpu"

DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Device:", DEVICE, "(GPU available)" if torch.cuda.is_available() else "(GPU not available)")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)

Device: cuda (GPU available)
GPU: NVIDIA GeForce RTX 3070 Laptop GPU
CUDA: 12.1
PyTorch: 2.4.1+cu121


## 3. Experiment Configuration

The notebook starts in smoke-test mode with `MAX_TESTS = 1`, short audio, and low diffusion steps. After the first successful run, will increase `MAX_TESTS`, `AUDIO_LENGTH_IN_S`, and `AUDIOLDM2_STEPS` for final presentation samples.

In [8]:
RUN_AUDIOLDM2 = True
RUN_BARK = True
RUN_TANGOFLUX = True
MAX_TESTS = 50

AUDIO_LENGTH_IN_S = 5.0
AUDIOLDM2_STEPS = 10
TANGOFLUX_STEPS = 10
BARK_MAX_NEW_TOKENS = 260
NUM_WAVEFORMS_PER_PROMPT = 1


AUDIO_LDM1_REPO = 'cvssp/audioldm2'
AUDIO_LDM2_REPO = 'anhnct/audioldm2_gigaspeech'
BARK_REPO = 'suno/bark-small'

TANGOFLUX_REPO = 'declare-lab/TangoFlux'

# Prompts for speech_tests

In [5]:
import json
PROMPTS_PATH = Path("prompts.json")
with PROMPTS_PATH.open(encoding="utf-8") as prompts_file:
    speech_tests = json.load(prompts_file)

speech_tests_df = pd.DataFrame(speech_tests)
speech_tests_df.head()

,id,category,transcript,style
0,auctioneer_fast,Extreme Pacing & Cadence,Do I hear fifty? Fifty dollars to the gentlema...,"A male auctioneer speaking at breakneck speed,..."
1,hypnotist_slow,Extreme Pacing & Cadence,Your eyelids are becoming incredibly heavy. Wi...,"A woman speaking extremely slowly, with a smoo..."
2,out_of_breath,Extreme Pacing & Cadence,I ran... all the way here... I think we lost t...,"A young man speaking while panting heavily, ou..."
3,valley_girl_upspeak,Extreme Pacing & Cadence,So I was literally just walking down the stree...,A young woman speaking with heavy vocal fry an...
4,stuttering_nervous,Extreme Pacing & Cadence,I... I-I didn't mean to do it. It was a... it ...,"A young boy stuttering heavily, his voice trem..."


## 4. Helper Functions

In [6]:
def clear_gpu_memory():
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()


def to_numpy_audio(audio):
    if isinstance(audio, torch.Tensor):
        audio = audio.detach().cpu().numpy()
    audio = np.asarray(audio).squeeze()
    return normalize_audio(audio.astype(np.float32))


def save_generated_audio(model_name, prompt_id, audio, sr):
    model_dir = OUTPUT_DIR / model_name
    model_dir.mkdir(exist_ok=True)
    filename = f'{prompt_id}.wav'.replace('/', '_').replace(' ', '_')
    filepath = model_dir / filename
    save_audio(audio, sr, filepath)
    return filepath


def list_generated_audio_files(output_dir="outputs"):
    root = Path(output_dir)
    audio_files = []

    for path in root.rglob("*.wav"):
        relative_path = path.relative_to(root)

        if len(relative_path.parts) == 2:
            model_name = relative_path.parts[0]
            test_id = path.stem
        else:
            parts = path.stem.split("_", 1)
            if len(parts) != 2:
                continue
            model_name, test_id = parts

        audio_files.append({
            "model": model_name,
            "test_id": test_id,
            "path": path,
            "relative_path": relative_path,
        })

    return audio_files

## 5. Load Models

The first run downloads large pretrained checkpoints. Later runs reuse the local Hugging Face cache.

In [ ]:
def load_audioldm2():
    from diffusers import AudioLDM2Pipeline

    print('Loading AudioLDM2:', AUDIO_LDM2_REPO)
    pipe = AudioLDM2Pipeline.from_pretrained(AUDIO_LDM2_REPO, torch_dtype=DTYPE)
    pipe = pipe.to(DEVICE)
    pipe.enable_vae_slicing()
    return pipe

def load_bark():
    print("Loading Bark:", BARK_REPO)
    processor = AutoProcessor.from_pretrained(BARK_REPO)
    model = BarkModel.from_pretrained(BARK_REPO, torch_dtype=DTYPE)

    tokenizer = processor.tokenizer
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    model = model.to(DEVICE)
    model.eval()
    return processor, model

def load_tangoflux():
    from tangoflux import TangoFluxInference

    print('Loading TangoFlux:', TANGOFLUX_REPO)
    model = TangoFluxInference(name=TANGOFLUX_REPO, device=DEVICE)
    return model

model_status = {}

print('Model loader functions are ready. Models are loaded one at a time in the generation cells.')

Model loader functions are ready. Models are loaded one at a time in the generation cells.


## 6. Generate Audio

In [ ]:
metrics = MetricsCollector()
generated_rows = []

In [43]:

def build_generation_prompt(item):
    return f"{item['style']} Spoken words: {item['transcript']}"

def run_audioldm2_item(pipe, item):
    generator = torch.Generator(device=DEVICE).manual_seed(SEED)
    start = time.time()

    result = pipe(
        prompt=item['style'],
        transcription=item['transcript'],
        num_inference_steps=AUDIOLDM2_STEPS,
        audio_length_in_s=AUDIO_LENGTH_IN_S,
        num_waveforms_per_prompt=NUM_WAVEFORMS_PER_PROMPT,
        generator=generator,
        max_new_tokens=512
    )

    elapsed = time.time() - start
    audio = to_numpy_audio(result.audios[0])
    return audio, 16000, elapsed


def run_bark_item(processor, model, item):
    inputs = processor(
        item["transcript"],
        voice_preset="v2/en_speaker_6",
        return_tensors="pt",
    )
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    start = time.time()
    with torch.inference_mode():
        audio = model.generate(
            **inputs,
            semantic_max_new_tokens=BARK_MAX_NEW_TOKENS,
            pad_token_id=model.generation_config.pad_token_id,
        )
    elapsed = time.time() - start
    sr = model.generation_config.sample_rate
    audio = to_numpy_audio(audio)
    return audio, sr, elapsed

def run_tangoflux_item(model, item):
    start = time.time()
    audio = model.generate(
        prompt=build_generation_prompt(item),
        steps=TANGOFLUX_STEPS,
        duration=AUDIO_LENGTH_IN_S,
        guidance_scale=3.5,
    )
    elapsed = time.time() - start
    
    # Handle both tensor and numpy array outputs
    if isinstance(audio, torch.Tensor):
        audio = audio.detach().cpu().numpy()
    
    audio = np.asarray(audio).squeeze()
    if audio.ndim == 2:
        audio = audio.mean(axis=0)  # stereo to mono
    audio = normalize_audio(audio.astype(np.float32))
    return audio, 44100, elapsed

def record_result(model_name, item, audio, sr, elapsed):
    filepath = save_generated_audio(model_name, item['id'], audio, sr)
    metrics.add_model_result(model_name, item['id'], elapsed, audio, sr)

    generated_rows.append({
        'model': model_name,
        'prompt_id': item['id'],
        'transcript': item['transcript'],
        'style': item['style'],
        'seconds': round(elapsed, 2),
        'sample_rate': sr,
        'file': str(filepath)
    })
    print(f'{model_name} | {item["id"]}: {elapsed:.1f}s -> {filepath}')



In [9]:
if RUN_AUDIOLDM2:
    audioldm2_pipe = None
    try:
        audioldm2_pipe = load_audioldm2()
        model_status['AudioLDM2'] = 'ready'

        selected_tests = speech_tests[:MAX_TESTS]

        for item in selected_tests:
            audio, sr, elapsed = run_audioldm2_item(audioldm2_pipe, item)
            record_result('AudioLDM2', item, audio, sr, elapsed)
    except Exception as error:
        model_status['AudioLDM2'] = str(error)
        print('AudioLDM2 failed:', error)
    finally:
        del audioldm2_pipe
        clear_gpu_memory()

Loading AudioLDM2: anhnct/audioldm2_gigaspeech


vae\diffusion_pytorch_model.safetensors not found
Loading pipeline components...:   0%|          | 0/11 [00:00<?, ?it/s]An error occurred while trying to fetch C:\Users\doman\.cache\huggingface\hub\models--anhnct--audioldm2_gigaspeech\snapshots\c812a7861f38a69441a8e0428438e782d9864614\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\doman\.cache\huggingface\hub\models--anhnct--audioldm2_gigaspeech\snapshots\c812a7861f38a69441a8e0428438e782d9864614\vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Loading pipeline components...:  82%|████████▏ | 9/11 [00:01<00:00,  5.06it/s]An error occurred while trying to fetch C:\Users\doman\.cache\huggingface\hub\models--anhnct--audioldm2_gigaspeech\snapshots\c812a7861f38a69441a8e0428438e782d9864614\projection_model: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\doman\.cache\huggingface\hub\models--anhnct--audioldm2_gigaspeech\sn

AudioLDM2 | auctioneer_fast: 11.3s -> outputs\AudioLDM2\auctioneer_fast.wav


100%|██████████| 10/10 [00:00<00:00, 14.35it/s]


AudioLDM2 | hypnotist_slow: 11.0s -> outputs\AudioLDM2\hypnotist_slow.wav


...

AudioLDM2 | dying_last_words: 11.9s -> outputs\AudioLDM2\dying_last_words.wav


100%|██████████| 10/10 [00:00<00:00, 15.41it/s]


AudioLDM2 | loud_sneezing: 11.8s -> outputs\AudioLDM2\loud_sneezing.wav


100%|██████████| 10/10 [00:00<00:00, 12.64it/s]


AudioLDM2 | intense_meditation: 12.2s -> outputs\AudioLDM2\intense_meditation.wav


In [45]:
if RUN_BARK:
    try:
        bark_processor, bark_model = load_bark()
        model_status['Bark'] = 'ready'
        selected_tests = speech_tests[:MAX_TESTS]
        for item in selected_tests:
            print(f'Starting Bark | {item["id"]}', flush=True)
            audio, sr, elapsed = run_bark_item(bark_processor, bark_model, item)
            record_result('Bark', item, audio, sr, elapsed)
    except Exception as error:
        model_status['Bark'] = str(error)
        print('Bark failed:', error)
    finally:
        del bark_model
        del bark_processor
        clear_gpu_memory()

Loading Bark: suno/bark-small
Starting Bark | auctioneer_fast
Bark | auctioneer_fast: 11.8s -> outputs\Bark\auctioneer_fast.wav
Starting Bark | hypnotist_slow
Bark | hypnotist_slow: 10.7s -> outputs\Bark\hypnotist_slow.wav
Starting Bark | out_of_breath
Bark | out_of_breath: 11.0s -> outputs\Bark\out_of_breath.wav
Starting Bark | valley_girl_upspeak
Bark | valley_girl_upspeak: 10.2s -> outputs\Bark\valley_girl_upspeak.wav
Starting Bark | stuttering_nervous
Bark | stuttering_nervous: 12.2s -> outputs\Bark\stuttering_nervous.wav
Starting Bark | drill_sergeant
Bark | drill_sergeant: 9.9s -> outputs\Bark\drill_sergeant.wav
Starting Bark | asmr_whisper
Bark | asmr_whisper: 12.1s -> outputs\Bark\asmr_whisper.wav
Starting Bark | epic_movie_trailer
Bark | epic_movie_trailer: 10.5s -> outputs\Bark\epic_movie_trailer.wav
Starting Bark | standup_comedian
Bark | standup_comedian: 10.2s -> outputs\Bark\standup_comedian.wav
Starting Bark | flight_attendant
Bark | flight_attendant: 9.7s -> outputs\Bar

In [ ]:
import os
# Without below disablement causes issues with tango flux
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

In [49]:
if RUN_TANGOFLUX:
    tangoflux_model = None
    try:
        tangoflux_model = load_tangoflux()
        model_status['TangoFlux'] = 'ready'

        #selected_tests = speech_tests[:1]
        selected_tests = speech_tests[:MAX_TESTS]

        for item in selected_tests:
            audio, sr, elapsed = run_tangoflux_item(tangoflux_model, item)
            record_result('TangoFlux', item, audio, sr, elapsed)
    except Exception as error:
        model_status['TangoFlux'] = str(error)
        print('TangoFlux failed:', error)
    finally:
        del tangoflux_model
        clear_gpu_memory()

Loading TangoFlux: declare-lab/TangoFlux


  0%|          | 0/10 [00:01<?, ?it/s]


TangoFlux | auctioneer_fast: 25.9s -> outputs\TangoFlux\auctioneer_fast.wav


  0%|          | 0/10 [00:01<?, ?it/s]


TangoFlux | hypnotist_slow: 26.0s -> outputs\TangoFlux\hypnotist_slow.wav


...

TangoFlux | dying_last_words: 46.7s -> outputs\TangoFlux\dying_last_words.wav


  0%|          | 0/10 [00:01<?, ?it/s]


TangoFlux | loud_sneezing: 45.8s -> outputs\TangoFlux\loud_sneezing.wav


  0%|          | 0/10 [00:01<?, ?it/s]


TangoFlux | intense_meditation: 46.4s -> outputs\TangoFlux\intense_meditation.wav


## 7. Listen to Generated Samples

In [3]:
from pathlib import Path

model_names = ['AudioLDM2', 'Bark', 'TangoFlux']
output_dir = Path('outputs')

print('Number of Generated WAV files per model:')

for model_name in model_names:
    model_dir = output_dir / model_name
    count = len(list(model_dir.glob('*.wav'))) if model_dir.exists() else 0
    print(f'{model_name}: {count} files')

Number of Generated WAV files per model:
AudioLDM2: 50 files
Bark: 50 files
TangoFlux: 50 files


In [4]:
from pathlib import Path
from IPython.display import Audio, display

model_names = ['AudioLDM2', 'Bark', 'TangoFlux']
output_dir = Path('outputs')
wav_name = 'ancient_wizard.wav'

for model_name in model_names:
    wav_path = output_dir / model_name / wav_name

    if wav_path.exists():
        print(model_name)
        display(Audio(filename=str(wav_path)))
    else:
        print(f'{model_name}: missing {wav_path}')

AudioLDM2


Bark


TangoFlux
